In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

In [ ]:
import anndata as ad
from adjustText import adjust_text

from cellassign import assign_cats

from cellbender.remove_background.downstream import load_anndata_from_input_and_output as load_anndata_cellbender

import cellrank as cr
from cellrank.estimators import GPCCA

import doubletdetection

from fa2 import ForceAtlas2

import gc

import harmonypy as hm

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

import networkx as nx

import numpy as np

import palantir

import pandas as pd

import phate

import plotly.express as px

from pybiomart import Server

import re 

from rpy2.robjects import globalenv
from rpy2.robjects import pandas2ri

import scanpy as sc
import scanpy.external as sce

import scFates as scf

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

import scipy.sparse as sp
from scipy.sparse import csr_matrix, issparse

import scvelo as scv

import seaborn as sns

from sklearn.decomposition import PCA

import triku as tk
import os, subprocess



In [ ]:
import sys

sys.path.append('..')

from pyfuncs.io import load_full_adata, add_ensembl_ids
from pyfuncs.dropletQC import classify_empty_and_damaged
from pyfuncs.general import preprocessing_adata_sub
from pyfuncs.plot_functions import magma, set_plotting_style, plot_volcano, plot_cell_stats, plot_gene_stats, savefig
from pyfuncs.common_vars import BASE_DIR, SEED, CELLBENDER_FIXED_ARGS
set_plotting_style()

In [ ]:
from pyfuncs.qc import  MT_CONTIG_MOUSE_REFSEQ, compute_qc_metrics, add_droplet_qc, flag_doublets, qc_embedding, plot_qc_overview, nf_band_report, ambient_top_genes, apply_qc_flags, qc_summary
from pyfuncs.normalization import concat_samples, preliminary_clusters, scran_size_factors, apply_size_factors, compare_normalizations, size_factor_report
from pyfuncs.processing import select_hvgs_preliminary, build_embeddings, select_hvgs_triku, soup_vs_hvg_report, harmony_merge_report
from pyfuncs.characterization import subset_and_reprocess, check_marker_dict, population_composition, reprocess_in_place
from pyfuncs.cell_types import DICT_RENAMING_INTEGRATION, PALETTE_CELL_TYPE_INTEGRATION

In [ ]:

from pyfuncs.interactions import run_liana, filter_liana, annotate_liana, plot_liana

In [ ]:
DATA_DIR_AA = f"{BASE_DIR}/data/ARAUZO_03/"
DATA_DIR_NI = f"{BASE_DIR}/data/public_scRNAseq/datasets/nicoletti_2023_GSE221736"
DATA_DIR_SO = f"{BASE_DIR}/data/public_scRNAseq/datasets/song_2023_GSE215922"
DATA_DIR_SU = f"{BASE_DIR}/data/public_scRNAseq/datasets/southerland_2023_GSE227075"


In [ ]:
GRUPO = "cell_type"     # la partición sobre la que se define PAGA
LOTE  = "gsm"

# Globinas: 45% del transcriptoma en algunas muestras, y 100% spliced.
# Fuera de velocity_genes sí o sí.
HB = ["Hbb-bs", "Hba-a1", "Hba-a2", "Hbb-bt", "Alas2", "Ahsp", "Bpgm"]

In [ ]:
from datetime import date
TODAY = str(date.today())

FIG_DIR = f"{BASE_DIR}/figures/{TODAY}/"
RESULTS_DIR = f"{BASE_DIR}/results/{TODAY}/"

In [ ]:
adata_AA = sc.read(f"{DATA_DIR_AA}/processed_adatas/AA_inhouse_dataset_processed.h5ad")
adata_NI = sc.read(f"{DATA_DIR_NI}/processed_adatas/NI_nicoletti_processed.h5ad")
adata_SO = sc.read(f"{DATA_DIR_SO}/processed_adatas/SO_song_processed.h5ad")
adata_SU = sc.read(f"{DATA_DIR_SU}/processed_adatas/SU_southerland_processed.h5ad")


In [ ]:
# We filter to get only the control data, we are not interested in the rest.

for adata in adata_AA, adata_NI, adata_SO, adata_SU:
    print(adata.obs["condition"].cat.categories)

adata_NI = adata_NI[adata_NI.obs["condition"] == "non_DEN"]
adata_SO = adata_SO[adata_SO.obs["condition"] == "sham"]
adata_SU = adata_SU[adata_SU.obs["condition"] == "C57BL6_sham"]

In [ ]:


# We apply this 
DICT_RENAMING_INTEGRATION = {
    'FAP_A': 'FAP.1',
    'FAP_AB': 'FAP.1',
    'FAP_AF': 'FAP.1',
    'FAP_B1': 'FAP.4',    
    'FAP_B2': 'FAP.4',
    'FAP_E': 'FAP.2',
    'FAP_C': 'FAP.3',
    'FAP_D': 'FAP.3',
    'FAP_F': 'FAP.3',
    'Krano_A': 'FAP.5',
    'Krano_B': 'FAP.6',
    'Krano_C': 'FAP.6',
    'Endothelial*': 'ENDO',
    'Sat_A1': 'SAT', 
    'Sat_A2': 'SAT', 
    'Sat_B': 'SAT', 
    'Teno_A': 'TNMD', 
    'Teno_D': 'TNMD',
    'Teno_B': 'TNMD', 
    'Teno_C': 'TNMD', 

    # Other populations
    "Immune": "IMM.MONO-MAC", 
    "B_cell": "IMM.B", 
    "Dendritic": "IMM.DEN",
    "Mast_cell": "IMM.MAST",
    "T_NK": "IMM.NK",
    "Neutrophil": "IMM.NEU",

    'Sat_U': 'SAT', 
    'Krano_U': 'FAP.5',
    "SMC": "SMC-SMMC",
    "SMMC": "SMC-SMMC",
    "Glial_Schwann": "GLIA",
    "Lymphatic_EC": "ENDO.LYMPH",
    'Pericyte': 'ENDO.PERI',
                               
}




PALETTE_CELL_TYPE_INTEGRATION = {
    # ---- Minor (más oscuros) ----
    # FAP (7)
    'FAP.1':"#A1D9F0",
    'FAP.2' :"#4C92AD",
    'FAP.3':"#12485E",
    'FAP.4':"#B2A1F0",
    'FAP.5' :"#694CAD",
    'FAP.6':"#26125E",

    "TNMD":"#6F5214",
    'SAT':"#2E9277",
    'SMC-SMMC':      "#79DD6F",
    'GLIA':      "#CAC031",

    'ENDO':   "#C23C64",
    'ENDO.LYMPH':   "#F8608D",
    'ENDO.PERI':   "#991F43",

    'IMM.MONO-MAC':      "#331903",
    'IMM.B':      "#D36B17",
    'IMM.NK':      "#F7A765",
    'IMM.DEN':      "#96490A",
    'IMM.MAST':      "#D35217",
    'IMM.NEU':      "#92390F",
}

In [ ]:
for adata in adata_AA, adata_NI, adata_SO, adata_SU:
    adata.obs["cell_type"] = adata.obs["minor_population"].astype(str).replace(DICT_RENAMING_INTEGRATION)
    adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
    adata.uns["cell_type_colors"] = [PALETTE_CELL_TYPE_INTEGRATION[i] for i in adata.obs["cell_type"].cat.categories]


In [ ]:
sc.pl.umap(adata_AA, color="cell_type")
sc.pl.umap(adata_NI, color="cell_type")
sc.pl.umap(adata_SO, color="cell_type")
sc.pl.umap(adata_SU, color="cell_type")

In [ ]:
adata_joint = sc.AnnData.concatenate(*[adata_AA, adata_NI, adata_SO, adata_SU], 
                                     batch_key="dataset", 
                                     batch_categories=["Fuertes", "Nicoletti", "Song", "Southerland"] )

In [ ]:
reprocess_in_place(adata_joint, batch_key="gsm", n_comps=40, )

In [ ]:
adata_joint.uns["cell_type_colors"] = [PALETTE_CELL_TYPE_INTEGRATION[i] for i in adata_joint.obs["cell_type"].cat.categories]
sc.pl.umap(adata_joint, color=["dataset",])
sc.pl.umap(adata_joint, color=["cell_type"])

In [ ]:
sc.pl.umap(adata_joint, color=["cell_type"], legend_loc="on data", legend_fontsize=4, legend_fontoutline=2)

# Interaction table

We are going to use the whole table of interactions and then filter by type of interaction.

In [ ]:
liana_df  = annotate_liana(run_liana(adata_joint, "cell_type", n_perms=3000, n_jobs=20), adata_joint, "cell_type")

In [ ]:
liana_df_AA  = annotate_liana(run_liana(adata_AA, "cell_type", n_perms=3000, n_jobs=20), adata_AA, "cell_type")

In [ ]:
liana_df_NI  = annotate_liana(run_liana(adata_NI, "cell_type", n_perms=3000, n_jobs=20), adata_NI, "cell_type")

In [ ]:
liana_df_SO  = annotate_liana(run_liana(adata_SO, "cell_type", n_perms=3000, n_jobs=20), adata_SO, "cell_type")

In [ ]:
liana_df_SU  = annotate_liana(run_liana(adata_SU, "cell_type", n_perms=3000, n_jobs=20), adata_SU, "cell_type")

## FAP - others

In [ ]:
FAP_POPS = ["FAP.1", "FAP.2", "FAP.3", "FAP.4", "FAP.5", "FAP.6"]

def filter_liana_pops(df, filter_type="FAP_FAP"): # FAP_FAP or FAP_OTHER
    if filter_type == "FAP_OTHER":
        return df[(df["source"].isin(FAP_POPS) & ~ df["target"].isin(FAP_POPS))  | (~ df["source"].isin(FAP_POPS) & df["target"].isin(FAP_POPS)) ].sort_values(by="spec_weight_oe", ascending=False)
    elif filter_type == "FAP_FAP":
        return df[(df["source"].isin(FAP_POPS)) & (df["target"].isin(FAP_POPS))].sort_values(by="spec_weight_oe", ascending=False)

In [ ]:
liana_df_FAP_others = filter_liana_pops(liana_df, filter_type="FAP_OTHER")
liana_df_FAP_others_AA = filter_liana_pops(liana_df_AA, filter_type="FAP_OTHER")
liana_df_FAP_others_NI = filter_liana_pops(liana_df_NI, filter_type="FAP_OTHER")
liana_df_FAP_others_SO = filter_liana_pops(liana_df_SO, filter_type="FAP_OTHER")
liana_df_FAP_others_SU = filter_liana_pops(liana_df_SU, filter_type="FAP_OTHER")

In [ ]:

def make_lr_label(row):
    ligand = str(row["ligand_complex"])
    receptor = str(row["receptor_complex"])
    if row["source"] in FAP_POPS:
        ligand = r"$\mathbf{" + ligand.replace("_", r"\_") + "}$"
    if row["target"] in FAP_POPS:
        receptor = r"$\mathbf{" + receptor.replace("_", r"\_") + "}$"
    return f"{ligand} → {receptor}"


def get_fap_direction(row):
    source_is_fap = row["source"] in FAP_POPS
    target_is_fap = row["target"] in FAP_POPS
    if source_is_fap and target_is_fap:
        return "FAP_to_FAP"
    if source_is_fap:
        return "FAP_to_other"
    if target_is_fap:
        return "other_to_FAP"
    return "other"


def _prepare(df_liana, population, specificity_rank_thresh,
             magnitude_rank_thres, spec_weight_oe_thresh, lr_logfc_thresh):
    required = ["source", "target", "ligand_complex", "receptor_complex",
                "spec_weight_oe", "specificity_rank", "magnitude_rank", "lr_logfc"]
    missing = [column for column in required if column not in df_liana]
    if missing:
        raise ValueError(f"Faltan columnas: {missing}")

    source_fap = df_liana["source"].isin(FAP_POPS)
    target_fap = df_liana["target"].isin(FAP_POPS)
    between_faps = population == "FAP" or population in FAP_POPS
    if between_faps:
        mask = source_fap & target_fap
        if population != "FAP":
            mask &= (df_liana["source"].eq(population)
                     | df_liana["target"].eq(population))
    else:
        mask = ((source_fap & df_liana["target"].eq(population))
                | (target_fap & df_liana["source"].eq(population)))

    # Copiar antes de asignar columnas evita modificar vistas del original.
    frame = df_liana.loc[mask].copy()
    if frame[["source", "target", "ligand_complex", "receptor_complex"]].isna().any().any():
        raise ValueError("Hay identidades de interacción incompletas.")
    for column in ["source", "target", "ligand_complex", "receptor_complex"]:
        frame[column] = frame[column].astype(str)

    frame["log_spec_weight_oe"] = np.log10(
        frame["spec_weight_oe"].where(frame["spec_weight_oe"] > 0)
    )
    frame["specificity_score"] = 1 - frame["specificity_rank"]
    frame["magnitude_score"] = 1 - frame["magnitude_rank"]

    if between_faps:
        frame["direction"] = "FAP_to_FAP"
        # Cada dirección es una columna distinta, incluida la autocrina.
        frame["FAP_population"] = frame["source"] + " → " + frame["target"]
        present_pairs = set(frame["FAP_population"])
        x_order = [f"{source} → {target}"
                   for source in FAP_POPS for target in FAP_POPS
                   if f"{source} → {target}" in present_pairs]
    else:
        source_is_fap = frame["source"].isin(FAP_POPS)
        frame["direction"] = np.where(source_is_fap, "FAP_to_other", "other_to_FAP")
        frame["FAP_population"] = frame["source"].where(source_is_fap, frame["target"])
        x_order = FAP_POPS

    frame["LR_name"] = frame["ligand_complex"] + "-" + frame["receptor_complex"]
    # Mismo identificador para un LR en todas las columnas comparadas.
    frame["LR_id"] = frame["LR_name"] + "__" + frame["direction"]
    frame["LR_label"] = pd.Series(
        [make_lr_label(row) for _, row in frame.iterrows()],
        index=frame.index, dtype=object,
    )
    frame["FAP_population"] = pd.Categorical(
        frame["FAP_population"], categories=x_order, ordered=True
    )
    frame["is_interesting"] = (
        (frame["specificity_rank"] < specificity_rank_thresh)
        & (frame["magnitude_rank"] < magnitude_rank_thres)
        & (frame["spec_weight_oe"] > spec_weight_oe_thresh)
        & (frame["lr_logfc"] > lr_logfc_thresh)
    ).fillna(False)
    return frame


def get_df_pop(df_liana, population, specificity_rank_thresh=0.05,
               magnitude_rank_thres=0.05, spec_weight_oe_thresh=10,
               lr_logfc_thresh=0.5, circle_size_var="spec_weight_oe"):
    """Elige LR que pasan los filtros en al menos una columna y conserva
    todas sus columnas disponibles para compararlas. Devuelve los mismos
    tres objetos que la función original: df_plot, label_map, lr_order.
    """
    frame = _prepare(df_liana, population, specificity_rank_thresh,
                     magnitude_rank_thres, spec_weight_oe_thresh, lr_logfc_thresh)
    if circle_size_var not in frame:
        raise ValueError(f"No existe la métrica {circle_size_var!r}.")
    interesting_ids = frame.loc[frame["is_interesting"], "LR_id"].unique()
    # Seleccionar por ID, no por etiqueta visual: emisión y recepción son distintas.
    frame = frame.loc[frame["LR_id"].isin(interesting_ids)].copy()
    lr_order = (
        frame.groupby("LR_id", observed=True)[circle_size_var]
        .agg(["max", "mean"])
        .sort_values(["max", "mean"], ascending=False, kind="stable")
        .index.tolist()
    )
    frame["LR_id"] = pd.Categorical(frame["LR_id"], categories=lr_order, ordered=True)
    label_map = (frame[["LR_id", "LR_label"]].drop_duplicates()
                 .set_index("LR_id")["LR_label"])
    return frame, label_map, lr_order


def get_df_pop_fixed_lr(df_liana, population, lr_order,
                        specificity_rank_thresh=0.05, magnitude_rank_thres=0.05,
                        spec_weight_oe_thresh=10, lr_logfc_thresh=0.5):
    """Mantiene los LR y su orden de una selección previa del mismo modo.
    Los filtros sólo recalculan los asteriscos; no vuelven a seleccionar LR.
    """
    frame = _prepare(df_liana, population, specificity_rank_thresh,
                     magnitude_rank_thres, spec_weight_oe_thresh, lr_logfc_thresh)
    lr_order = list(lr_order)
    frame = frame.loc[frame["LR_id"].isin(lr_order)].copy()
    frame["LR_id"] = pd.Categorical(frame["LR_id"], categories=lr_order, ordered=True)
    return frame




def plot_liana_pop(df_plot, label_map, lr_order,
                   circle_size_var="spec_weight_oe", circle_hue_var="lr_logfc",
                   size_lim=(5, 500), hue_lim=(-4, 4), figsize=None,
                   shade_fap_blocks=True, block_color="0.94"):
    """Dibuja la tabla preparada. Fija coordenadas numéricas para que los
    asteriscos y los ticks coincidan incluso si faltan categorías intermedias.
    En FAP → FAP sombrea los bloques de FAP.1, FAP.3 y FAP.5 emisoras.
    El sombreado respeta las identidades aunque falten pares en un dataset.
    """
    lr_order = list(lr_order)
    if len(set(lr_order)) != len(lr_order):
        raise ValueError("lr_order contiene IDs repetidos.")
    if isinstance(df_plot["FAP_population"].dtype, pd.CategoricalDtype):
        x_order = df_plot["FAP_population"].cat.categories.tolist()
    else:
        x_order = df_plot["FAP_population"].dropna().unique().tolist()
    frame = df_plot.copy()
    if frame.duplicated(["FAP_population", "LR_id"]).any():
        raise ValueError(
            "Hay varias filas para el mismo LR y columna. Separa los datasets "
            "o muestras antes de dibujar; los puntos se solaparían."
        )
    frame["_plot_x"] = frame["FAP_population"].astype(object).map(
        {name: i for i, name in enumerate(x_order)}
    )
    frame["_plot_y"] = frame["LR_id"].astype(object).map(
        {name: i for i, name in enumerate(lr_order)}
    )
    if frame[["_plot_x", "_plot_y"]].isna().any().any():
        raise ValueError("Hay categorías que no están en el orden de los ejes.")
    if figsize is None:
        figsize = (max(5, 0.48 * len(x_order) + 2), 12)
    fig, ax = plt.subplots(figsize=figsize)
    if shade_fap_blocks:
        shaded_sources = set(FAP_POPS[::2])
        # Agrupar columnas consecutivas conserva las bandas incluso si el
        # usuario reordena categorías o faltan algunos pares dirigidos.
        start = 0
        while start < len(x_order):
            source, separator, target = str(x_order[start]).partition(" → ")
            if separator and source in shaded_sources and target in FAP_POPS:
                end = start + 1
                while end < len(x_order):
                    next_source, next_sep, next_target = str(x_order[end]).partition(" → ")
                    if not (next_sep and next_source == source and next_target in FAP_POPS):
                        break
                    end += 1
                ax.axvspan(start - 0.5, end - 0.5, facecolor=block_color,
                           edgecolor="none", zorder=0)
                start = end
            else:
                start += 1
    if not frame.empty:
        # Seaborn omite los puntos sin tamaño/color; tampoco dibujar su asterisco.
        valid = np.isfinite(frame[circle_size_var]) & np.isfinite(frame[circle_hue_var])
        visible = frame.loc[valid]
        if not visible.empty:
            sns.scatterplot(
                data=visible, x="_plot_x", y="_plot_y",
                size=circle_size_var, hue=circle_hue_var, sizes=size_lim,
                palette="RdBu_r", hue_norm=hue_lim,
                edgecolor="black", linewidth=0.3, ax=ax,
            )
            for _, row in visible.loc[visible["is_interesting"]].iterrows():
                ax.annotate("*", (row["_plot_x"], row["_plot_y"]),
                            xytext=(0, -2), textcoords="offset points",
                            ha="center", va="center", fontsize=8,
                            fontweight="bold", color="black", zorder=10)
            ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left",
                      borderaxespad=0, frameon=False)
    else:
        ax.text(0.5, 0.5, "No hay interacciones para mostrar",
                transform=ax.transAxes, ha="center", va="center")

    ax.set_xticks(range(len(x_order)), labels=x_order, rotation=45, ha="right")
    ax.set_yticks(range(len(lr_order)),
                  labels=[label_map.get(lr_id, lr_id) for lr_id in lr_order])
    ax.set_xlim(-0.5, max(len(x_order) - 0.5, 0.5))
    ax.set_ylim(max(len(lr_order) - 0.5, 0.5), -0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.tight_layout()
    return fig

In [ ]:
liana_df_FAP_others

In [ ]:
def plot_general_case(df_liana, population, specificity, magnitude, spec_weight, lr_logfc, 
                      circle_size_var, circle_hue_var, 
                      size_lim=(5, 400), hue_lim=(-4, 4), figsize=(4, 12), ytitle=0.93):
    
    df_plot, label_map, lr_order = get_df_pop(df_liana, population, 
                                                specificity_rank_thresh=specificity,
                                                magnitude_rank_thres=magnitude, 
                                                spec_weight_oe_thresh=spec_weight, 
                                                lr_logfc_thresh=lr_logfc, 
                                                circle_size_var=circle_size_var,)
    fig = plot_liana_pop(df_plot, label_map, lr_order, 
                        circle_size_var=circle_size_var, circle_hue_var=circle_hue_var, 
                        size_lim=size_lim, hue_lim=hue_lim, figsize=figsize)
    fig.suptitle(f"L-R interactions between $\mathbf{{FAPs}}$ and {population}", y=ytitle)

    return df_plot, label_map, lr_order, fig


def plot_specific_case(df_liana_dataset, dataset_name, population, lr_order, label_map,
                       specificity, magnitude, spec_weight, lr_logfc, 
                       circle_size_var, circle_hue_var, 
                       size_lim_general=(5, 400), hue_lim_general=(-4, 4), figsize_general=(4, 12), ytitle_general=0.93, 
                       size_lim_specific=(5, 400), hue_lim_specific=(-4, 4), figsize_specific=(4, 12), ytitle_specific=0.93):
    
    df_plot_dataset_general = get_df_pop_fixed_lr(
        df_liana_dataset,
        population=population,
        lr_order=lr_order,
        )
    fig_general = plot_liana_pop(df_plot_dataset_general, label_map, lr_order, 
                        circle_size_var=circle_size_var, circle_hue_var=circle_hue_var, 
                        size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general)
    fig_general.suptitle(f"L-R interactions between $\mathbf{{FAPs}}$ and {population} - {dataset_name}\nTop global interactions", y=ytitle_general)


    
    df_plot_dataset_specific, label_map_dataset, lr_order_dataset = get_df_pop(df_liana_dataset, 
                                                                               population=population, 
                                                specificity_rank_thresh=specificity,
                                                magnitude_rank_thres=magnitude, 
                                                spec_weight_oe_thresh=spec_weight, 
                                                lr_logfc_thresh=lr_logfc, circle_size_var=circle_size_var,)
    fig_specific = plot_liana_pop(df_plot_dataset_specific, label_map_dataset, lr_order_dataset, 
                        circle_size_var=circle_size_var, circle_hue_var=circle_hue_var, 
                        size_lim=size_lim_specific, hue_lim=hue_lim_specific, figsize=figsize_specific)
    fig_specific.suptitle(f"L-R interactions between $\mathbf{{FAPs}}$ and {population} - {dataset_name}\nTop specific interactions", y=ytitle_specific)

    return df_plot_dataset_specific, label_map, lr_order, fig_general, fig_specific


In [ ]:
dict_datasets = {"Fuertes": liana_df_FAP_others_AA, 
                 "Nicoletti": liana_df_FAP_others_NI, 
                 "Song": liana_df_FAP_others_SO, 
                 "Southerland": liana_df_FAP_others_SU}

CIRCLE_SIZE = "spec_weight_oe"
CIRCLE_COLOR = "lr_logfc"




In [ ]:
adata_joint.obs["cell_type"].cat.categories

In [ ]:
filter_liana(liana_df_FAP_others, genes=["Mag"]).sort_values("spec_weight_oe")

In [ ]:
POP_CHOICE = "ENDO"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=30
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    print(dataset)
    df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                        SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                        CIRCLE_SIZE, CIRCLE_COLOR, 
                        size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                        size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                        figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                        ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

    savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    plt.show()



In [ ]:
POP_CHOICE = "ENDO.LYMPH"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.9
MAGNITUDE=0.9
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "ENDO.PERI"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.7
MAGNITUDE=0.7
SPEC_WEIGHT=20
LR_LOGFC=0.7


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]


fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "GLIA"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.B"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 6)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.DEN"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1



# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.MAST"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1

# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.MONO-MAC"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()


# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.NEU"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "IMM.NK"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(12, 10)
ytitle_general=0.95


dict_figparams = {"Fuertes": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (12, 6), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (12, 6), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1

# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "SAT"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(6, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (6, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (6, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (6, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (6, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "SMC-SMMC"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1


# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
POP_CHOICE = "TNMD"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(4, 12)
ytitle_general=0.95

dict_figparams = {"Fuertes": {"figsize_specific": (4, 6), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (4, 8), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (4, 10), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (4, 10), "ytitle_specific": 0.99}, }

SPECIFICITY=0.8
MAGNITUDE=0.8
SPEC_WEIGHT=25
LR_LOGFC=1

# Tune in filtering parameters
df_liana = liana_df_FAP_others

df_liana_pop = df_liana[(df_liana["source"] == POP_CHOICE) | (df_liana["target"] == POP_CHOICE)]
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_others, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

# Apply to the populations
for dataset, liana_df_FAP_others_dataset in dict_datasets.items():
    if POP_CHOICE in liana_df_FAP_others_dataset[["target", "source"]].values.sum():
        print(dataset)
        df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_others_dataset, dataset, POP_CHOICE, lr_order, label_map,
                            SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                            CIRCLE_SIZE, CIRCLE_COLOR, 
                            size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                            size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                            figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                            ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

        savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
        plt.show()



In [ ]:
df_plot

## FAP - others

In [ ]:
liana_df_FAP_FAP = filter_liana_pops(liana_df, filter_type="FAP_FAP")
liana_df_FAP_FAP_AA = filter_liana_pops(liana_df_AA, filter_type="FAP_FAP")
liana_df_FAP_FAP_NI = filter_liana_pops(liana_df_NI, filter_type="FAP_FAP")
liana_df_FAP_FAP_SO = filter_liana_pops(liana_df_SO, filter_type="FAP_FAP")
liana_df_FAP_FAP_SU = filter_liana_pops(liana_df_SU, filter_type="FAP_FAP")

In [ ]:
POP_CHOICE = "FAP"

size_lim_general = (5, 400)
hue_lim_general=(-4, 4)
figsize_general=(18, 16)
ytitle_general=1

dict_figparams = {"Fuertes": {"figsize_specific": (18, 16), "ytitle_specific": 0.99}, 
                 "Nicoletti": {"figsize_specific": (18, 16), "ytitle_specific": 0.99},  
                 "Song": {"figsize_specific": (18, 16), "ytitle_specific": 0.99},  
                 "Southerland": {"figsize_specific": (18, 16), "ytitle_specific": 0.99}, }

SPECIFICITY=0.85
MAGNITUDE=0.85
SPEC_WEIGHT=15
LR_LOGFC=0.7



# Tune in filtering parameters
df_liana_pop = liana_df_FAP_FAP
df_liana_pop["log_spec_weight_oe"] = np.log10(df_liana_pop["spec_weight_oe"])
df_liana_pop["specificity_score"] = 1 - df_liana_pop["specificity_rank"]
df_liana_pop["magnitude_score"] = 1 - df_liana_pop["magnitude_rank"]

fig, axs = plt.subplots(1, 4, figsize=(12, 3))
sns.kdeplot(df_liana_pop, x= "log_spec_weight_oe", ax=axs[0], cut=0)
axs[0].axvline(np.log10(SPEC_WEIGHT))
sns.kdeplot(df_liana_pop, x= "lr_logfc", ax=axs[1], cut=0)
axs[1].axvline((LR_LOGFC))
sns.kdeplot(df_liana_pop, x= "specificity_rank", ax=axs[2], cut=0)
axs[2].axvline((SPECIFICITY))
sns.kdeplot(df_liana_pop, x= "magnitude_rank", ax=axs[3], cut=0)
axs[3].axvline((MAGNITUDE))
plt.tight_layout()



# Apply to the general case
df_plot, label_map, lr_order, fig = plot_general_case(liana_df_FAP_FAP, 
                                                    POP_CHOICE, SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC, 
                    circle_size_var=CIRCLE_SIZE, circle_hue_var=CIRCLE_COLOR, 
                    size_lim=size_lim_general, hue_lim=hue_lim_general, figsize=figsize_general, ytitle=ytitle_general)

savefig(fig=fig, filename=f"4_LIANA_dotplot_INTEGRATED_{POP_CHOICE}", fig_dir=FIG_DIR)
plt.show()

display(df_plot)

# Apply to the populations
for dataset, liana_df_FAP_FAP_dataset in dict_datasets.items():
    print(dataset)
    df_plot_dataset, label_map_dataset, lr_order_dataset, fig_general_dataset, fig_specific_dataset = plot_specific_case(liana_df_FAP_FAP_dataset, dataset, POP_CHOICE, lr_order, label_map,
                        SPECIFICITY, MAGNITUDE, SPEC_WEIGHT, LR_LOGFC,
                        CIRCLE_SIZE, CIRCLE_COLOR, 
                        size_lim_general=size_lim_general, hue_lim_general=hue_lim_general, figsize_general=figsize_general, ytitle_general=ytitle_general, 
                        size_lim_specific=size_lim_general, hue_lim_specific=hue_lim_general, 
                        figsize_specific=dict_figparams[dataset]["figsize_specific"], 
                        ytitle_specific=dict_figparams[dataset]["ytitle_specific"])

    savefig(fig=fig_general_dataset, filename=f"4_LIANA_dotplot_INTEGRATED-{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    savefig(fig=fig_specific_dataset, filename=f"4_LIANA_dotplot_{dataset}_{POP_CHOICE}", fig_dir=FIG_DIR)
    plt.show()



In [ ]:
liana_df_FAP_FAP